In [10]:
import os, glob, time, random, requests
from pathlib import Path
from tqdm import tqdm

In [11]:
type_name = 'zingers_raspberry__888109010089__6'

In [12]:
# ===== CONFIG =====
API_KEY = "EG5BE30T0zEXlkeuR5hn"           # your Roboflow API key
DATASET_NAME = "products_annotations-rc8gz"    # your dataset slug
FOLDER = f"/home/ubuntu/additional_drive/shwan_data/scripts/all_frames/{type_name}/"         # path to image folder
IMAGE_PATTERN = "*g"                       # *.jpg / *.png / *g etc.
SEED = 42
MAX_RETRIES = 3
DELAY_BASE = 1.0
TAG_WITH_FOLDER = True                     # adds folder name as Roboflow tag
# ===================

upload_url = f"https://api.roboflow.com/dataset/{DATASET_NAME}/upload"
session = requests.Session()
random.seed(SEED)


In [13]:
r = requests.get("https://api.roboflow.com/", params={"api_key": API_KEY})
print(r.status_code, r.text)


200 {
    "welcome": "Welcome to the Roboflow API.",
    "instructions": "You are successfully authenticated.",
    "docs": "https://docs.roboflow.com",
    "workspace": "annotations-swude"
}


In [14]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

folder = Path(FOLDER).resolve()
files = [Path(p) for p in glob.glob(str(folder / IMAGE_PATTERN))]
files = [f for f in files if f.suffix.lower() in IMG_EXTS]
if not files:
    raise SystemExit("No image files found.")

random.shuffle(files)
n = len(files)
train = files[:int(0.8 * n)]
valid = files[int(0.8 * n):int(0.9 * n)]
test  = files[int(0.9 * n):]
print(f"Total: {n} | train: {len(train)} | valid: {len(valid)} | test: {len(test)}")


Total: 60 | train: 48 | valid: 6 | test: 6


In [15]:
def upload_one(path, split, tags=None):
    params = {"api_key": API_KEY}
    data = {"name": path.name, "split": split}
    if tags:
        data["tag"] = tags

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            with open(path, "rb") as f:
                r = session.post(upload_url, params=params, files={"file": f}, data=data, timeout=120)
            try:
                j = r.json()
            except ValueError:
                j = r.text
            if r.status_code == 200:
                return True, j
            else:
                if attempt < MAX_RETRIES:
                    time.sleep(DELAY_BASE * (2 ** (attempt - 1)))
                else:
                    return False, f"HTTP {r.status_code}: {j}"
        except requests.RequestException as e:
            if attempt < MAX_RETRIES:
                time.sleep(DELAY_BASE * (2 ** (attempt - 1)))
            else:
                return False, str(e)


In [16]:
successes, failures = 0, 0
tags = [folder.name] if TAG_WITH_FOLDER else None

for split_name, split_files in [("train", train), ("valid", valid), ("test", test)]:
    print(f"\nUploading {split_name} ({len(split_files)} files)...")
    for p in tqdm(split_files):
        ok, resp = upload_one(p, split_name, tags=tags)
        if ok:
            successes += 1
        else:
            failures += 1
            print(f"[FAIL] {p.name} -> {resp}")

print("\n=== Summary ===")
print(f"Successes: {successes}")
print(f"Failures : {failures}")



Uploading train (48 files)...


100%|██████████| 48/48 [05:44<00:00,  7.17s/it]



Uploading valid (6 files)...


100%|██████████| 6/6 [00:43<00:00,  7.26s/it]



Uploading test (6 files)...


100%|██████████| 6/6 [00:46<00:00,  7.82s/it]


=== Summary ===
Successes: 60
Failures : 0
